# Notebook 2 — SVM de padrões de ritmo

**Meta:** classificar **padrões de ritmo** separados das condições diagnósticas. Ritmo sinusal (`SR`) é uma classe **explícita**, não sinônimo de ECG completamente normal; `AFIB`, `SBRAD`, `STACH`, `SARRH` etc. são outros possíveis padrões.

**Arquitetura:** um sistema multirrótulo de **SVMs binários independentes por código SCP de ritmo**. Evitamos forçar SR, bradicardia, taquicardia e fibrilação atrial a se excluírem artificialmente. O modelo não realiza diagnóstico clínico validado.

**Dados:** CSVs da PTB-XL fornecidos; sinais completos da base WFDB devem ser disponibilizados localmente para treinar.
**Caminho configurado no Windows:** `C:\Users\Matheus\Documents\PTB-XL`. Os metadados serão lidos dessa pasta e os sinais localizados em `records500/` pelos caminhos do CSV. Os resultados serão gravados em `resultados_svm_reestruturados/` dentro da PTB-XL.

**Destino dos modelos treinados:** `C:\Users\Matheus\Documents\Modelos\modelos_02`. O cache intermediário permanece na pasta PTB-XL.


**Versão contextualizada:** incorpora indicadores de faixa etária, sexo, aplicabilidade das regras e critérios verificáveis, com rastreabilidade das limitações de medição.

## 1. Fluxo completo e limites metodológicos

```mermaid
flowchart TD
 A[CSV de metadados + SCP] --> B[Auditoria de códigos, idade, sexo e pacientes]
 B --> C[Separação oficial por paciente: folds 1-8 / 9 / 10]
 C --> D[Leitura dos sinais WFDB em 12 derivações]
 D --> E[Controle determinístico de qualidade + extração por derivação]
 E --> F[Características numéricas + idade + sexo + indicadores de faltantes]
 F --> G[Imputação e padronização ajustadas somente no treino]
 G --> H[SVMs binários por rótulo, sem exclusividade artificial]
 H --> I[Ajuste de parâmetros e limiares no fold 9]
 I --> J[Teste final no fold 10]
 J --> K[Relatórios, matrizes, ROC/PR e modelo joblib]
```

**Importante:** o conjunto CSV não inclui os sinais digitais. Para treinar, a pasta `records500/` da PTB-XL precisa estar acessível. Sem ela, apenas a auditoria dos rótulos pode ser executada. O notebook **não declara nenhum resultado antes de executar a avaliação real**.

**Interpretação clínica:** os resultados são associações estatísticas com anotações SCP, não diagnósticos automáticos validados. As regras por idade/sexo/derivação são usadas como contexto e para auditoria. Não se aplicam automaticamente limiares clínicos sem protocolo validado; PR, QT e ST diagnósticos não são inventados quando não medidos com segurança.

## 2. Dependências e reprodutibilidade

Instalar no mesmo ambiente Python: `pip install numpy pandas scipy scikit-learn wfdb joblib matplotlib nbformat`.

O código fixa a semente, reutiliza o `strat_fold`, nunca usa o fold 10 para escolher parâmetros, utiliza uma imputação aprendida apenas no treino e registra resultados por rótulo. A divisão por paciente é verificada explicitamente.

In [1]:
from pathlib import Path
import ast, json, warnings
import numpy as np
import pandas as pd
from scipy import signal as sps
import matplotlib.pyplot as plt
import wfdb, joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             balanced_accuracy_score, roc_auc_score, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from sklearn.base import clone

SEED = 42
BASE = Path(r"C:\Users\Matheus\Documents\PTB-XL")  # Caminho absoluto no Windows
CSV_METADATA = BASE / "ptbxl_database.csv"
CSV_SCP = BASE / "scp_statements.csv"
OUT = BASE / "resultados_svm_reestruturados"
OUT.mkdir(parents=True, exist_ok=True)
# Destino separado dos modelos treinados e relatórios de avaliação.
MODELOS_DIR = Path(r"C:\Users\Matheus\Documents\Modelos\modelos_02")
MODELOS_DIR.mkdir(parents=True, exist_ok=True)
print("Os modelos serão exportados para:", MODELOS_DIR)
# 0 = todos os registros; >0 = amostra estratificada por fold para testes rápidos
MAX_REGISTROS = 0
# SVC-RBF custa caro; cap opcional por classe apenas NO TREINO de cada rótulo
MAX_TREINO_POR_CLASSE = 2500
MIN_POS_TREINO = 30
MIN_POS_VALID = 5
MIN_POS_TESTE = 5

assert BASE.is_dir(), f"Pasta PTB-XL não encontrada: {BASE}"
assert CSV_METADATA.is_file() and CSV_SCP.is_file(), f"CSVs não encontrados em {BASE}"
df = pd.read_csv(CSV_METADATA)
scp = pd.read_csv(CSV_SCP, index_col=0)
print('Metadados:', df.shape, '| SCP:', scp.shape)
print('Pastas com sinais presentes:', (BASE / 'records500').exists(), (BASE / 'records100').exists())

Os modelos serão exportados para: C:\Users\Matheus\Documents\Modelos\modelos_02
Metadados: (21799, 28) | SCP: (71, 12)
Pastas com sinais presentes: True True


## 3. Auditoria dos códigos SCP e integridade dos folds

- Os pesos armazenados em `scp_codes` **não** são usados como medidas clínicas. A presença de uma chave indica uma anotação; o peso original permanece preservado para auditoria.
- Registros com anotação `NORM` juntamente com MI/STTC/CD/HYP recebem sinalização de conflito. Para o treinamento proposto, eles serão removidos **conservadoramente**; a quantidade excluída é reportada.
- Um registro pode ter vários rótulos positivos. Nunca é transformado numa classe única por prioridade.
- A informação `SR` representa ritmo sinusal; não é sinônimo de ausência de qualquer condição.
- Os registros sem códigos aproveitáveis não são tratados automaticamente como normais.

In [2]:
def parse_scp(v):
    try:
        d = ast.literal_eval(v) if isinstance(v,str) else v
        return {str(k).strip():float(x) for k,x in d.items()} if isinstance(d,dict) else {}
    except (ValueError, SyntaxError, TypeError):
        return {}

df['codes'] = df['scp_codes'].map(parse_scp)
df['code_set'] = df['codes'].map(lambda d:set(d))
assert not (df.groupby('patient_id', dropna=True)['strat_fold'].nunique()>1).any(), 'Paciente aparece em vários folds!'
DIAGNOSTIC = {'MI','STTC','CD','HYP'}
def category_map(scp):
    maps={}
    for key,row in scp.iterrows():
        cats=set()
        d=row.get('diagnostic_class')
        if pd.notna(d) and str(d) in DIAGNOSTIC: cats.add(str(d))
        if pd.notna(row.get('form')) and float(row['form'])==1: cats.add('FORM')
        if pd.notna(row.get('rhythm')) and float(row['rhythm'])==1: cats.add('RHYTHM')
        maps[str(key)] = cats
    return maps
CODE_TO_CATEGORY=category_map(scp)
df['categories'] = df.code_set.map(lambda codes:set().union(*(CODE_TO_CATEGORY.get(c,set()) for c in codes)))
df['conflict_norm_diagnostic'] = df.apply(lambda r:'NORM' in r.code_set and bool(r.categories & DIAGNOSTIC),axis=1)
print('Registros:',len(df),'| conflitos NORM + grupo diagnóstico:',int(df.conflict_norm_diagnostic.sum()))
print('Contagens por fold:\n',df.strat_fold.value_counts().sort_index().to_string())
print('Códigos de ritmo:',[k for k,v in CODE_TO_CATEGORY.items() if 'RHYTHM' in v])

Registros: 21799 | conflitos NORM + grupo diagnóstico: 445
Contagens por fold:
 strat_fold
1     2175
2     2181
3     2192
4     2174
5     2174
6     2173
7     2176
8     2173
9     2183
10    2198
Códigos de ritmo: ['SR', 'AFIB', 'STACH', 'SARRH', 'SBRAD', 'PACE', 'SVARR', 'BIGU', 'AFLT', 'SVTAC', 'PSVT', 'TRIGU']


## 4. Pré-processamento por registro: preservando as 12 derivações

**Mudança em relação ao script antigo:** em vez de reunir 120 janelas (12 × 10 s) numa mediana única, cada derivação tem características próprias, com nomes como `II_std`, `V2_energia` e `V6_amp_pp`. O intervalo completo de 10 segundos é mantido para análise do ritmo.

Características abaixo são **proxies computacionais**: média, desvio, amplitude pico-a-pico, energia, derivada robusta e energia em bandas. Elas **não** substituem a mensuração clínica de ST, PR, QRS ou QT. O extrator de RR é heurístico e fornece frequência/regularidade somente quando existem picos plausíveis; os casos duvidosos tornam-se `NaN` e são auditáveis. Não se inferem ondas P nem critérios de infarto a partir desses proxies.

**Qualidade:** recusa sinais vazios, canais insuficientes e amplitudes não finitas; não elimina registros devido ao diagnóstico ou ao erro do modelo. Dados ausentes são imputados no pipeline dentro do treino. O sexo é mantido como campo codificado no banco; não o reinterpretamos como sexo biológico confirmado por outras fontes.

In [3]:
LEADS = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
def clean_1d(x,fs):
    x=np.asarray(x,dtype=float)
    if len(x)<int(3*fs) or np.mean(np.isfinite(x))<0.99: return None
    x=np.where(np.isfinite(x),x,np.nanmedian(x[np.isfinite(x)]))
    # Filtragem 0.5-40 Hz SOMENTE para características de forma e detecção de picos.
    # Por remover componentes lentos, esta filtragem NÃO é adequada para medir desvio ST clínico.
    lo=max(0.5, min(0.5,fs/8)); hi=min(40.,fs*.4)
    if hi<=lo: return None
    sos=sps.butter(3,[lo,hi],btype='bandpass',fs=fs,output='sos')
    return sps.sosfiltfilt(sos,x)

def rpeaks_heuristic(x,fs):
    # Detector experimental; não validado como delineador clínico de QRS.
    if x is None:return np.array([],dtype=int)
    sos=sps.butter(2,[5,min(20,fs*.4)],btype='bandpass',fs=fs,output='sos')
    v=sps.sosfiltfilt(sos,x)
    e=np.convolve(np.gradient(v)**2, np.ones(max(3,int(.08*fs)))/max(3,int(.08*fs)),mode='same')
    peaks,_=sps.find_peaks(e,distance=int(.28*fs),prominence=max(np.percentile(e,85)*.3,1e-12))
    refined=[]
    for p in peaks:
        l=max(0,p-int(.075*fs));r=min(len(v),p+int(.075*fs)+1)
        if r>l: refined.append(l+int(np.argmax(np.abs(v[l:r]))))
    refined=np.unique(refined)
    if len(refined)>1: refined=refined[np.r_[True,np.diff(refined)>int(.24*fs)]]
    return refined

def lead_features(raw,fs,prefix):
    out={}
    x=clean_1d(raw,fs)
    cols=['std','rms','ptp','mad','deriv_mad','energy_1_5','energy_5_15','energy_15_35']
    if x is None:
        out.update({f'{prefix}_{k}':np.nan for k in cols})
        out[f'{prefix}_valid']=0
        return out
    out[f'{prefix}_valid']=1
    out[f'{prefix}_std']=float(np.std(x))
    out[f'{prefix}_rms']=float(np.sqrt(np.mean(x*x)))
    out[f'{prefix}_ptp']=float(np.ptp(x))
    out[f'{prefix}_mad']=float(np.median(np.abs(x-np.median(x))))
    out[f'{prefix}_deriv_mad']=float(np.median(np.abs(np.diff(x))))
    freq,psd=sps.welch(x,fs=fs,nperseg=min(len(x),int(2*fs)))
    for a,b,name in [(1,5,'energy_1_5'),(5,15,'energy_5_15'),(15,35,'energy_15_35')]:
        mask=(freq>=a)&(freq<b)
        out[f'{prefix}_{name}']=float(np.trapz(psd[mask],freq[mask])) if mask.sum()>1 else np.nan
    return out

def rhythm_features(signal,fs,lead_names):
    # Usa derivação II, com fallback I se II não existir; resultados são estimativas, não laudo.
    names=list(lead_names)
    target='II' if 'II' in names else ('I' if 'I' in names else None)
    vals={k:np.nan for k in ['n_peaks','hr_bpm','rr_mean_ms','rr_sd_ms','rr_cv','rmssd_ms','rr_p10_ms','rr_p90_ms']}
    if target is None:return vals
    x=clean_1d(signal[:,names.index(target)],fs)
    p=rpeaks_heuristic(x,fs)
    rr=np.diff(p)/fs if len(p)>=2 else np.array([])
    rr=rr[(rr>=.28)&(rr<=2.5)]
    if len(rr)<4:return vals
    vals.update(n_peaks=float(len(p)),hr_bpm=float(60/np.mean(rr)),rr_mean_ms=float(1000*np.mean(rr)),
                rr_sd_ms=float(1000*np.std(rr)),rr_cv=float(np.std(rr)/np.mean(rr)),
                rmssd_ms=float(1000*np.sqrt(np.mean(np.diff(rr)**2))) if len(rr)>1 else np.nan,
                rr_p10_ms=float(1000*np.quantile(rr,.1)),rr_p90_ms=float(1000*np.quantile(rr,.9)))
    return vals

def record_features(record,meta):
    x=np.asarray(record.p_signal,dtype=float)
    fs=float(record.fs)
    if x.ndim!=2 or x.shape[0]<int(5*fs):return None
    names=list(record.sig_name)
    out={'age':pd.to_numeric(meta.get('age'),errors='coerce'),
         'sex_ptbxl':pd.to_numeric(meta.get('sex'),errors='coerce')}
    for lead in LEADS:
        if lead in names:out.update(lead_features(x[:,names.index(lead)],fs,lead))
        else:out.update(lead_features(np.array([]),fs,lead))
    out.update(rhythm_features(x,fs,names))
    return out
print('Extrator de características definido: 12 derivações preservadas + parâmetros RR experimentais.')

Extrator de características definido: 12 derivações preservadas + parâmetros RR experimentais.


## 5. Processar sinais, guardar cache e auditar ausências

Uma linha por ECG. O cache evita reprocessar milhares de arquivos ao alterar apenas os hiperparâmetros. **Apague o cache se alterar o extrator ou a definição da amostra.** A chave inclui o nome da tarefa/versão. Não permita que o cache antigo com outra engenharia de características seja reutilizado.
**Tratamento de falhas (revisão v2):** caches vazios não são reutilizados; se nenhum registro puder ser processado, o notebook mostra as principais causas e grava `cache_*.falhas.csv`. A integração espectral usa `np.trapz`, compatível com versões anteriores do NumPy.


In [4]:
def select_metadata(frame):
    d=frame.copy()
    if MAX_REGISTROS and len(d)>MAX_REGISTROS:
        per=max(1,MAX_REGISTROS//d.strat_fold.nunique())
        d=(d.groupby('strat_fold',group_keys=False)
             .apply(lambda p:p.sample(n=min(len(p),per),random_state=SEED)))
    return d

def build_feature_table(meta, cache_path):
    if cache_path.exists():
        cached = pd.read_pickle(cache_path)
        if isinstance(cached, pd.DataFrame) and not cached.empty and 'ecg_id' in cached.columns:
            print('Cache recuperado:',cache_path, '| ECGs:',len(cached))
            return cached
        print('AVISO: cache vazio/inválido:',cache_path)
        print('Descartando o cache e processando novamente para identificar a causa original.')
        cache_path.unlink()

    rows=[];fail=[]
    for j,(_,r) in enumerate(meta.iterrows(),1):
        possible=[r.get('filename_hr'),r.get('filename_lr')]
        record=None
        for name in possible:
            if not isinstance(name,str):continue
            prefix=BASE/name
            if not prefix.with_suffix('.hea').exists():continue
            try:
                record=wfdb.rdrecord(str(prefix)); break
            except Exception as exc:
                last_error=str(exc)
        if record is None:
            fail.append((r.ecg_id,'arquivo ausente ou ilegível'));continue
        try: feat=record_features(record,r)
        except Exception as exc:
            fail.append((r.ecg_id,f'caracteristicas: {exc}'));continue
        if feat is None:fail.append((r.ecg_id,'sinal inválido ou curto'));continue
        feat.update(ecg_id=int(r.ecg_id),patient_id=r.patient_id,strat_fold=int(r.strat_fold))
        rows.append(feat)
        if j%500==0:print(f'{j}/{len(meta)} processados; aproveitados {len(rows)}',flush=True)
    pd.DataFrame(fail,columns=['ecg_id','motivo']).to_csv(cache_path.with_suffix('.falhas.csv'),index=False)
    result=pd.DataFrame(rows)
    if result.empty:
        exemplos = pd.Series([motivo for _,motivo in fail]).value_counts().head(8).to_dict()
        raise RuntimeError(
            'Nenhum ECG gerou características válidas. '
            f'Total de candidatos: {len(meta)}. '
            f'Principais causas: {exemplos}. '
            f'Consulte o relatório: {cache_path.with_suffix(".falhas.csv")}. '
            'Verifique se existem os arquivos .hea/.dat de records500 e se o extrator '
            'é compatível com suas versões de NumPy/SciPy.'
        )
    if 'ecg_id' not in result.columns:
        raise RuntimeError('Erro interno: coluna ecg_id ausente no resultado da extração.')
    result.to_pickle(cache_path)
    print('ECGs aproveitados:',len(result),'| falhas:',len(fail))
    return result

## Integração de critérios clínicos condicionados a idade, sexo e derivação

Este notebook incorpora um **motor contextual de características**, baseado nos dois arquivos fornecidos (`regras_sbc.py` e `st_multiderivacao.py`), sem pressupor que os limites representem diagnósticos automáticos.

**Faixas da interface:** pediátrico (0–16), adulto jovem (17–39), adulto (40–59), idoso (60+). Atenção: a faixa 17–39 contém pacientes com 17 anos; as regras exclusivas para adultos usam **idade ≥18**, independentemente da faixa selecionada.

**Sexo no CSV PTB-XL:** `0 = masculino`, `1 = feminino`. Essa codificação não deve ser confundida com valores textuais selecionados no visualizador.

**Critérios documentados e aplicabilidade:**

| Medida | Critério disponibilizado | Restrição |
|---|---|---|
| FC | 50–99 bpm, referência numérica de frequência sinusal | Não demonstra ritmo sinusal; não é diagnóstico |
| PR | >200 ms | Adulto, FC 50–90, medição PR válida, confirmação P:QRS pendente |
| QRS | ≥120 ms | Duração isolada não identifica bloqueio nem seu ramo |
| QTc Bazett | >450 ms masculino / >470 ms feminino | Adulto, FC 60–90, QT/RR confiáveis |
| ST em V2–V3 | F: 0,15 mV; M <40: 0,25 mV; M ≥40: 0,20 mV | Adultos, ponto J medido e validado; verificar derivações contíguas |
| ST em outras derivações elegíveis | 0,10 mV | Adultos e análise anatômica/qualidade |

**Limitação que não pode ser omitida:** o extrator original dos notebooks calcula estatísticas por derivação e RR, mas **não fornece medidas validadas de PR, duração do QRS, QTc ou ST**. O motor contextual mantém essas regras *não avaliáveis* quando faltam medidas; não inventa valores. Não se transforma ausência de medição em ausência de doença.

As características contextuais serão aprendidas pelo SVM a partir dos rótulos SCP; as regras são apenas *indicadores explicativos e condicionais*, sem alteração artificial do alvo. A variação da pontuação ao mudar idade ou sexo é possível, mas **não obrigatória** e necessita validação por subgrupos.

**Fonte de implementação:** `regras_sbc.py` e `st_multiderivacao.py` fornecidos pelo usuário, com referências à SBC 2022 e ESC 2023. A portabilidade do motor é experimental; a medição de ST do código original continua não validada clinicamente.


In [5]:

# ================================================================
# MOTOR CONTEXTUAL EXPERIMENTAL (sem criar diagnósticos artificiais)
# Regras baseadas nos arquivos fornecidos:
#   regras_sbc.py / st_multiderivacao.py
# NOTA: somente uma medição confiável poderá ativar uma comparação clínica.
# ================================================================
CONTEXT_RULES_VERSION = "contexto_sbc_st_v1"
AGE_GROUPS = ("pediatrico_0_16", "adulto_jovem_17_39",
              "adulto_40_59", "idoso_60_mais")

def age_group_label(age):
    if pd.isna(age) or not (0 <= age <= 120):
        return "nao_informado"
    age = float(age)
    if age <= 16:
        return AGE_GROUPS[0]
    if age < 40:
        return AGE_GROUPS[1]
    if age < 60:
        return AGE_GROUPS[2]
    return AGE_GROUPS[3]

def sex_from_ptbxl(sex):
    """Na PTB-XL: sex=0 -> masculino; sex=1 -> feminino."""
    if pd.isna(sex):
        return None
    return "M" if sex == 0 else ("F" if sex == 1 else None)

def st_elevation_threshold_mv(lead, age, sex):
    """Porta fiel à regra de limiares ADULTOS do st_multiderivacao.py.
    NÃO mede ST nem confirma IAM; contiguidade/morfologia seguem pendentes.
    """
    if pd.isna(age) or float(age) < 18:
        return np.nan
    if lead not in LEADS or lead == "aVR":
        return np.nan
    sex = sex_from_ptbxl(sex)
    if lead in ("V2", "V3"):
        if sex == "F": return 0.15
        if sex == "M": return 0.25 if float(age) < 40 else 0.20
        return np.nan
    return 0.10

def contextualize_features(table):
    """Recebe extração por ECG e gera mesmas colunas no treino e inferência.
    Sem criar rótulos a partir de regras. Ausente != regra normal.
    """
    d=table.copy()
    age=pd.to_numeric(d["age"],errors="coerce")
    sex=pd.to_numeric(d["sex_ptbxl"],errors="coerce")
    d["idade_ausente"]=age.isna().astype(int)
    d["sexo_ausente"]=~sex.isin([0,1])
    d["sexo_ausente"]=d["sexo_ausente"].astype(int)
    d["adulto_maior_igual_18"]=(age>=18).astype(int)
    # Mantemos idade contínua para aprender interações suaves entre faixas.
    groups=age.map(age_group_label)
    for group in AGE_GROUPS:
        d["faixa_"+group]=(groups==group).astype(int)
    d["faixa_indeterminada"]=(groups=="nao_informado").astype(int)
    d["sexo_masculino"]=(sex==0).astype(int)
    d["sexo_feminino"]=(sex==1).astype(int)
    d["sexo_desconhecido"]=~sex.isin([0,1])
    d["sexo_desconhecido"]=d["sexo_desconhecido"].astype(int)
    # Interações explícitas com FC e RR, mas não forçam patologia com idade/sexo.
    hr=pd.to_numeric(d.get("hr_bpm",pd.Series(np.nan,index=d.index)),errors="coerce")
    d["fc_abaixo_50_numerico"]=np.where(hr.notna(),(hr<50).astype(float),np.nan)
    d["fc_acima_99_numerico"]=np.where(hr.notna(),(hr>99).astype(float),np.nan)
    d["fc_fora_50_99_numerico"]=np.where(hr.notna(),((hr<50)|(hr>99)).astype(float),np.nan)
    d["fc_ref_mensuravel"]=hr.notna().astype(int)
    # Só descreve faixa de FC; SR/ritmo sinusal não foi confirmado automaticamente.

    # Exemplo concreto de regra idade+sexo+derivação, somente quando medida ST está presente.
    # O extrator original NÃO fornece 'V2_st_j_mv' nem 'V3_st_j_mv'.
    # Portanto, sem um delineador validado, a comparação fica SEM MEDIÇÃO.
    for lead in ["V2","V3"]:
        st_col=f"{lead}_st_j_mv"
        measured=pd.to_numeric(d[st_col],errors="coerce") if st_col in d.columns else pd.Series(np.nan,index=d.index)
        thresholds=pd.Series([st_elevation_threshold_mv(lead,a,s) for a,s in zip(age,sex)],index=d.index,dtype=float)
        eligible=measured.notna() & thresholds.notna()
        d[f"regra_{lead}_st_j_aplicavel"]=eligible.astype(int)
        d[f"regra_{lead}_st_j_excesso_mv"]=np.where(eligible,measured-thresholds,np.nan)
        d[f"regra_{lead}_st_j_limiar_atendido"]=np.where(eligible,(measured>=thresholds).astype(float),np.nan)

    # PR, QRS e QTc: somente avaliar quando a medida EXPLICITAMENTE existir.
    pr=pd.to_numeric(d["pr_ms"],errors="coerce") if "pr_ms" in d else pd.Series(np.nan,index=d.index)
    qrs=pd.to_numeric(d["qrs_ms"],errors="coerce") if "qrs_ms" in d else pd.Series(np.nan,index=d.index)
    qtc=pd.to_numeric(d["qtc_bazett_ms"],errors="coerce") if "qtc_bazett_ms" in d else pd.Series(np.nan,index=d.index)
    pr_ok=(age>=18)&hr.between(50,90,inclusive="both")&pr.notna()
    d["regra_pr_avaliavel"]=pr_ok.astype(int)
    d["regra_pr_maior_200"]=np.where(pr_ok,(pr>200).astype(float),np.nan)
    qrs_ok=qrs.notna()
    d["regra_qrs_avaliavel"]=qrs_ok.astype(int)
    d["regra_qrs_maior_igual_120"]=np.where(qrs_ok,(qrs>=120).astype(float),np.nan)
    qtc_limit=pd.Series(np.where(sex==0,450.,np.where(sex==1,470.,np.nan)),index=d.index)
    qtc_ok=(age>=18)&hr.between(60,90,inclusive="both")&qtc.notna()&qtc_limit.notna()
    d["regra_qtc_avaliavel"]=qtc_ok.astype(int)
    d["regra_qtc_acima_limite"]=np.where(qtc_ok,(qtc>qtc_limit).astype(float),np.nan)
    d["regra_qtc_distancia_limite_ms"]=np.where(qtc_ok,qtc-qtc_limit,np.nan)
    return d

def contexto_paciente_relatorio(age, sex_ptbxl):
    """Metadados explícitos para o visualizador; não são diagnóstico."""
    return {
        "idade": None if pd.isna(age) else float(age),
        "sexo": sex_from_ptbxl(sex_ptbxl),
        "faixa": age_group_label(age),
        "st_v2_limiar_mv": None if pd.isna(st_elevation_threshold_mv("V2",age,sex_ptbxl))
                             else float(st_elevation_threshold_mv("V2",age,sex_ptbxl)),
        "st_v3_limiar_mv": None if pd.isna(st_elevation_threshold_mv("V3",age,sex_ptbxl))
                             else float(st_elevation_threshold_mv("V3",age,sex_ptbxl))
    }

print("Motor contextual definido:",CONTEXT_RULES_VERSION)


Motor contextual definido: contexto_sbc_st_v1


## 6. Treinamento: padronização, amostragem e busca simples de hiperparâmetros

Os modelos são **SVM binários independentes** por rótulo. `SimpleImputer` e `StandardScaler` são ajustados *dentro* do pipeline, apenas no treino. A amostragem opcional reduz o custo do SVC-RBF e ocorre **somente no treino**, preservando avaliação integral no fold 9/10. A seleção do par `(C, gamma)` e do limiar é feita no fold 9. O fold 10 é usado uma única vez para avaliação final. O SVM não aprende regras de conduta clínica: aprende a predizer rótulos SCP disponíveis.

In [6]:
def split_X(df_features,metadata,targets):
    if df_features.empty or 'ecg_id' not in df_features.columns:
        raise ValueError('A tabela de características está vazia ou sem ecg_id. '
                         'Exclua o cache antigo e execute novamente a extração; '
                         'consulte o arquivo .falhas.csv.')
    if 'ecg_id' not in metadata.columns:
        raise ValueError('O CSV de metadados não possui a coluna ecg_id.')
    if df_features['ecg_id'].duplicated().any():
        raise ValueError('A extração gerou ecg_id duplicados.')
    merged=df_features.merge(metadata[['ecg_id','code_set','categories','strat_fold','patient_id']],
                             on='ecg_id',how='inner',suffixes=('','_meta'),validate='one_to_one')
    # Fonte confiável: os folds e IDs da tabela de metadados
    if 'strat_fold_meta' in merged:assert (merged.strat_fold==merged.strat_fold_meta).all()
    featcols=[c for c in df_features.columns if c not in {'ecg_id','patient_id','strat_fold'}]
    featcols=[c for c in featcols if pd.api.types.is_numeric_dtype(merged[c])]
    return merged,featcols

def make_pipeline_svm(C=1,gamma='scale'):
    return Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),
                     ('scaler',StandardScaler()),
                     ('svm',SVC(kernel='rbf',C=C,gamma=gamma,class_weight='balanced',probability=False,random_state=SEED))])

def dec_scores(model,X):
    return np.asarray(model.decision_function(X)).ravel()

def find_threshold(y,score):
    from sklearn.metrics import precision_recall_curve
    pre,rec,thresholds=precision_recall_curve(y,score)
    f=2*pre[:-1]*rec[:-1]/np.maximum(pre[:-1]+rec[:-1],1e-12)
    if len(thresholds)==0:return 0.
    return float(thresholds[int(np.argmax(f))])

def sample_train_indices(y,index,cap,seed):
    rng=np.random.default_rng(seed)
    selection=[]
    for v in [0,1]:
        ids=index[y[index]==v]
        if cap and len(ids)>cap:ids=rng.choice(ids,size=cap,replace=False)
        selection.extend(ids.tolist())
    return np.array(sorted(selection),dtype=int)

def evaluate(y,score,threshold):
    pred=(score>=threshold).astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    return dict(n=int(len(y)),positivos=int(np.sum(y)),threshold=float(threshold),
                accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                precision_pos=float(precision_score(y,pred,zero_division=0)),
                recall_pos=float(recall_score(y,pred,zero_division=0)),
                specificity=float(tn/(tn+fp)) if tn+fp else np.nan,
                f1_pos=float(f1_score(y,pred,zero_division=0)),
                f1_macro=float(f1_score(y,pred,average='macro',zero_division=0)),
                roc_auc=float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan,
                pr_auc_ap=float(average_precision_score(y,score)) if len(np.unique(y))==2 else np.nan,
                tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

def train_targets(merged,feature_cols,labels,output_dir):
    output_dir.mkdir(parents=True,exist_ok=True)
    X=merged[feature_cols].to_numpy(dtype=float)
    fold=merged.strat_fold.to_numpy(dtype=int)
    tr=np.flatnonzero(fold<=8);va=np.flatnonzero(fold==9);te=np.flatnonzero(fold==10)
    assert len(tr)>0 and len(va)>0 and len(te)>0, 'Treino, validação e teste devem existir.'
    assert not (set(merged.iloc[tr].patient_id)&set(merged.iloc[te].patient_id))
    report=[]; skipped=[]
    for label,y in labels.items():
        y=np.asarray(y,dtype=int)
        if y[tr].sum()<MIN_POS_TREINO or y[va].sum()<MIN_POS_VALID or y[te].sum()<MIN_POS_TESTE or len(np.unique(y[tr]))<2 or len(np.unique(y[va]))<2:
            skipped.append({'label':label,'reason':'amostras positivas/negativas insuficientes','train_pos':int(y[tr].sum()),'val_pos':int(y[va].sum()),'test_pos':int(y[te].sum())});continue
        train_idx=sample_train_indices(y,tr,MAX_TREINO_POR_CLASSE,SEED)
        best=None
        for C,gamma in [(0.3,'scale'),(1.0,'scale'),(3.0,'scale')]:
            pipe=make_pipeline_svm(C,gamma)
            pipe.fit(X[train_idx],y[train_idx])
            score=dec_scores(pipe,X[va])
            threshold=find_threshold(y[va],score)
            metrics=evaluate(y[va],score,threshold)
            rank=metrics['f1_pos']
            if best is None or rank>best['rank']:
                best={'model':pipe,'C':C,'gamma':gamma,'threshold':threshold,'rank':rank,'val':metrics}
        # O fold 10 NÃO influencia nenhuma decisão de hiperparâmetro ou limiar.
        score=dec_scores(best['model'],X[te]);metrics=evaluate(y[te],score,best['threshold'])
        safe=''.join(c if c.isalnum() or c in '_-' else '_' for c in label)
        result={'label':label,'C':best['C'],'gamma':best['gamma'],'train_pos':int(y[tr].sum()),
                'validation':best['val'], 'test':metrics}
        report.append(result)
        # Exporta TODOS os classificadores elegíveis individualmente, em formato compactado.
        # O pipeline contém imputação, escalonamento e SVM já ajustados.
        model_path = output_dir / f'{safe}.joblib'
        joblib.dump({'model':best['model'],'threshold':best['threshold'],'label':label,
                     'features':list(feature_cols),'task':output_dir.name,
                     'context_rules_version':CONTEXT_RULES_VERSION,
                     'age_groups':list(AGE_GROUPS),
                     'C':best['C'],'gamma':best['gamma'],
                     'note':'score=decision_function; nao e probabilidade clinica'},
                    model_path,compress=3)
        result['arquivo_modelo'] = model_path.name
        cm=np.array([[metrics['tn'],metrics['fp']],[metrics['fn'],metrics['tp']]])
        fig,ax=plt.subplots(figsize=(4.4,3.8));ConfusionMatrixDisplay(cm,display_labels=['ausente','presente']).plot(ax=ax,colorbar=False)
        ax.set_title(label+' - fold 10');fig.tight_layout();fig.savefig(output_dir/f'{safe}_matriz.png',dpi=160);plt.close(fig)
        if len(np.unique(y[te]))==2:
            fpr,tpr,_=roc_curve(y[te],score)
            p,r,_=precision_recall_curve(y[te],score)
            fig,ax=plt.subplots(figsize=(5,4));ax.plot(fpr,tpr,label=f'ROC-AUC={metrics["roc_auc"]:.3f}');ax.plot([0,1],[0,1],linestyle='--');ax.legend();ax.set_title(label+' - ROC');fig.tight_layout();fig.savefig(output_dir/f'{safe}_roc.png',dpi=160);plt.close(fig)
            fig,ax=plt.subplots(figsize=(5,4));ax.plot(r,p,label=f'AP={metrics["pr_auc_ap"]:.3f}');ax.legend();ax.set_title(label+' - Precisão × sensibilidade');fig.tight_layout();fig.savefig(output_dir/f'{safe}_pr.png',dpi=160);plt.close(fig)
        print(label, '| validação F1:',round(best['rank'],3),'| teste F1:',round(metrics['f1_pos'],3),'| teste PR-AP:',round(metrics['pr_auc_ap'],3))
    pd.json_normalize(report).to_csv(output_dir/'metricas_por_rotulo.csv',index=False)
    pd.DataFrame(skipped).to_csv(output_dir/'rotulos_nao_treinados.csv',index=False)
    with open(output_dir/'configuracao.json','w',encoding='utf-8') as f:
        json.dump({'seed':SEED,'max_registros':MAX_REGISTROS,'cap_train_per_class':MAX_TREINO_POR_CLASSE,
                   'feature_cols':feature_cols,'context_rules_version':CONTEXT_RULES_VERSION,'label_names':list(labels),'train_folds':list(range(1,9)),
                   'validation_fold':9,'test_fold':10,'candidates_C':[.3,1,3]},f,indent=2,ensure_ascii=False)
    # Inventário verificável de TODOS os modelos realmente treinados.
    # Modelos não treinados não são exportados; veja rotulos_nao_treinados.csv.
    import zipfile
    files = [output_dir / item['arquivo_modelo'] for item in report]
    missing = [str(p) for p in files if not p.is_file()]
    if missing:
        raise FileNotFoundError(f'Faltam modelos esperados na exportação: {missing}')
    inventory = {
        'tarefa': output_dir.name,
        'quantidade_treinados': len(files),
        'quantidade_nao_treinados': len(skipped),
        'arquivos_modelo': [p.name for p in files],
        'limiares': {item['label']: item['validation']['threshold'] for item in report},
        'observacao': 'Contém apenas modelos treinados, com pipeline e limiar ajustados.'
    }
    inventory_path=output_dir/'inventario_modelos.json'
    inventory_path.write_text(json.dumps(inventory,ensure_ascii=False,indent=2),encoding='utf-8')
    archive=output_dir/f'modelos_svm_{output_dir.name}_completos.zip'
    with zipfile.ZipFile(archive, 'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in files: z.write(p, arcname=p.name)
        for fname in ['inventario_modelos.json','configuracao.json',
                      'metricas_por_rotulo.csv','rotulos_nao_treinados.csv']:
            p=output_dir/fname
            if p.exists(): z.write(p,arcname=p.name)
    print('Concluído:',len(report),'modelos exportados individualmente; ignorados:',len(skipped))
    print('Pasta:',output_dir.resolve())
    print('Pacote com todos os modelos treinados:',archive.resolve())
    return report,skipped

## 7. Definição dos alvos de ritmo

O arquivo `scp_statements.csv` identifica formalmente os códigos de ritmo por `rhythm == 1`. O notebook **inclui SR** e não mistura esses rótulos com `MI/STTC/CD/HYP/FORM` como uma única doença. As coocorrências são preservadas. Uma observação `NORM + SR` pode ser apropriada; `NORM + SBRAD` não será automaticamente classificada como doença ou removida por esse motivo.

**Cuidado:** o código SCP `PACE` significa ritmo de marcapasso, não simplesmente uma arritmia patológica; cada código requer interpretação própria. Rótulo ausente significa apenas sem anotação explícita, não ausência clínica definitivamente confirmada.

In [7]:
RHYTHM_CODES=sorted([str(k) for k,row in scp.iterrows()
                     if pd.notna(row.get('rhythm')) and float(row['rhythm'])==1.0])
print('Códigos de ritmo:',RHYTHM_CODES)
counts={k:int(df.code_set.map(lambda codes:k in codes).sum()) for k in RHYTHM_CODES}
print('Contagem bruta dos rótulos:',counts)
# Não remover NORM+SR; excluir somente discordâncias diagnósticas identificadas para alinhamento entre tarefas.
metadata=df.loc[~df.conflict_norm_diagnostic & df.code_set.map(bool)].copy()
metadata=select_metadata(metadata)
print('Candidatos para treinamento:',len(metadata))
print('Registros SR e com outra marcação de ritmo:',int(metadata.code_set.map(lambda s:'SR' in s and len(s & set(RHYTHM_CODES))>1).sum()))

Códigos de ritmo: ['AFIB', 'AFLT', 'BIGU', 'PACE', 'PSVT', 'SARRH', 'SBRAD', 'SR', 'STACH', 'SVARR', 'SVTAC', 'TRIGU']
Contagem bruta dos rótulos: {'AFIB': 1514, 'AFLT': 73, 'BIGU': 82, 'PACE': 294, 'PSVT': 24, 'SARRH': 772, 'SBRAD': 637, 'SR': 16748, 'STACH': 826, 'SVARR': 157, 'SVTAC': 27, 'TRIGU': 20}
Candidatos para treinamento: 21354
Registros SR e com outra marcação de ritmo: 61


## 8. Extração de características temporais para ritmo

Os parâmetros `hr_bpm`, `rr_mean_ms`, `rr_sd_ms`, `rr_cv`, `rmssd_ms` e quantis RR são estimativas de um detector automatizado em 10 segundos. A extração ocorre em II, com fallback para I. A heurística não é um detector de QRS clinicamente validado e **não calcula** ondas P, intervalo PR ou atividade fibrilatória; o relatório deve declarar isso. Se o detector falhar, as variáveis RR permanecem ausentes. As características por todas as 12 derivações continuam disponíveis para o SVM.

In [8]:
assert (BASE/'records500').exists() or (BASE/'records100').exists(), (
  'Não foram encontrados registros WFDB. Extraia records500/ da PTB-XL junto dos CSVs e ajuste BASE.')
features_table=build_feature_table(metadata,OUT/'cache_ritmos_v2_regras.pkl')
features_table=contextualize_features(features_table)
merged,feature_cols=split_X(features_table,metadata,RHYTHM_CODES)
labels={k:merged.code_set.map(lambda s:int(k in s)).to_numpy() for k in RHYTHM_CODES}
print('Registros:',len(merged),'| características:',len(feature_cols))
for k,y in labels.items():print(k, 'train:',int(y[merged.strat_fold<=8].sum()),
                                    'validation:',int(y[merged.strat_fold==9].sum()),
                                    'test:',int(y[merged.strat_fold==10].sum()))
print('Critérios contextuais integrados:', CONTEXT_RULES_VERSION)
print('Aplicabilidade ST V2:', int(merged['regra_V2_st_j_aplicavel'].sum()), 'de',len(merged))
print('Faixas etárias:', {g:int(merged['faixa_'+g].sum()) for g in AGE_GROUPS})


Cache recuperado: C:\Users\Matheus\Documents\PTB-XL\resultados_svm_reestruturados\cache_ritmos_v2_regras.pkl | ECGs: 21354
Registros: 21354 | características: 146
AFIB train: 1209 validation: 151 test: 152
AFLT train: 59 validation: 7 test: 7
BIGU train: 66 validation: 8 test: 8
PACE train: 237 validation: 29 test: 28
PSVT train: 19 validation: 3 test: 2
SARRH train: 597 validation: 70 test: 71
SBRAD train: 489 validation: 63 test: 59
SR train: 13113 validation: 1640 test: 1636
STACH train: 651 validation: 81 test: 81
SVARR train: 127 validation: 15 test: 14
SVTAC train: 21 validation: 3 test: 3
TRIGU train: 16 validation: 2 test: 2
Critérios contextuais integrados: contexto_sbc_st_v1
Aplicabilidade ST V2: 0 de 21354
Faixas etárias: {'pediatrico_0_16': 86, 'adulto_jovem_17_39': 2655, 'adulto_40_59': 6747, 'idoso_60_mais': 11574}


## 9. Exportação completa e compactada dos modelos

Cada classificador SVM **efetivamente treinado** gera um arquivo `<CODIGO>.joblib` com `compress=3`. O arquivo contém o pipeline completo (imputação, `StandardScaler` e SVC), o limiar escolhido na validação (fold 9), o código da classe, o nome e a ordem das características e os hiperparâmetros.

Após a avaliação, o notebook cria automaticamente:

- **Modelos individuais:** `<CODIGO>.joblib`, um por rótulo treinado.
- **Pacote completo:** `modelos_svm_modelos_02_completos.zip`, contendo todos os `.joblib` gerados, sem omitir nenhum.
- **Inventário:** `inventario_modelos.json`, com nomes e contagens.
- **Auditoria:** `rotulos_nao_treinados.csv`, indicando quais rótulos não satisfizeram os critérios mínimos; não é possível exportar modelos que não foram treinados.
- **Resultados:** `metricas_por_rotulo.csv` e `configuracao.json`.

A saída será criada em `C:\Users\Matheus\Documents\Modelos\modelos_02` quando o treinamento for executado no seu computador. **O ZIP não é um modelo único:** ele é um pacote com todos os modelos individuais e seus metadados. Os resultados são globais por ECG e não representam diagnósticos clínicos validados.

## 9. Treinar e avaliar os classificadores de ritmo

Cada código é um problema **presente × ausente** independente. Se não há exemplos suficientes, o classificador não é treinado e entra em `rotulos_nao_treinados.csv`. No fold 9 são escolhidos `C` e limiar; o fold 10 fornece métricas finais. Um alto escore de SR **não valida** a ausência de alterações MI/STTC/CD/HYP.

In [9]:
report, skipped=train_targets(merged,feature_cols,labels,MODELOS_DIR)
pd.DataFrame([{'rotulo':r['label'],**r['test']} for r in report]).round(3)

AFIB | validação F1: 0.701 | teste F1: 0.671 | teste PR-AP: 0.675
AFLT | validação F1: 0.833 | teste F1: 0.333 | teste PR-AP: 0.453
BIGU | validação F1: 0.462 | teste F1: 0.0 | teste PR-AP: 0.129
PACE | validação F1: 0.582 | teste F1: 0.667 | teste PR-AP: 0.615
SARRH | validação F1: 0.261 | teste F1: 0.261 | teste PR-AP: 0.143
SBRAD | validação F1: 0.557 | teste F1: 0.393 | teste PR-AP: 0.358
SR | validação F1: 0.91 | teste F1: 0.899 | teste PR-AP: 0.933
STACH | validação F1: 0.693 | teste F1: 0.729 | teste PR-AP: 0.782
SVARR | validação F1: 0.118 | teste F1: 0.0 | teste PR-AP: 0.011
Concluído: 9 modelos exportados individualmente; ignorados: 3
Pasta: C:\Users\Matheus\Documents\Modelos\modelos_02
Pacote com todos os modelos treinados: C:\Users\Matheus\Documents\Modelos\modelos_02\modelos_svm_modelos_02_completos.zip


,rotulo,n,positivos,threshold,accuracy,balanced_accuracy,precision_pos,recall_pos,specificity,f1_pos,f1_macro,roc_auc,pr_auc_ap,tn,fp,fn,tp
0,AFIB,2147,152,0.488,0.948,0.856,0.606,0.750,0.963,0.671,0.821,0.952,0.675,1921,74,38,114
1,AFLT,2147,7,0.932,0.996,0.642,0.400,0.286,0.999,0.333,0.666,0.926,0.453,2137,3,5,2
2,BIGU,2147,8,1.490,0.996,0.500,0.000,0.000,1.000,0.000,0.499,0.901,0.129,2139,0,8,0
3,PACE,2147,28,0.505,0.992,0.820,0.692,0.643,0.996,0.667,0.831,0.953,0.615,2111,8,10,18
4,SARRH,2147,71,0.908,0.918,0.686,0.186,0.437,0.934,0.261,0.609,0.849,0.143,1940,136,40,31
5,SBRAD,2147,59,1.092,0.967,0.687,0.397,0.390,0.983,0.393,0.688,0.926,0.358,2053,35,36,23
6,SR,2147,1636,-0.705,0.839,0.727,0.861,0.941,0.513,0.899,0.751,0.845,0.933,262,249,96,1540
7,STACH,2147,81,0.654,0.977,0.899,0.660,0.815,0.984,0.729,0.859,0.987,0.782,2032,34,15,66
8,SVARR,2147,14,1.734,0.993,0.500,0.000,0.000,0.999,0.000,0.498,0.523,0.011,2131,2,14,0


## 10. Diagrama e limitações específicas de ritmo

```mermaid
flowchart TD
 S[ECG 10 s, 12 derivações] --> P[Características de cada derivação]
 S --> RR[Detector heurístico QRS e série RR]
 P --> X[Vetor estruturado + idade + sexo]
 RR --> X
 X --> SR[SVM SR]
 X --> AFIB[SVM AFIB]
 X --> BRAD[SVM SBRAD]
 X --> TACH[SVM STACH]
 X --> O[Outros SVMs de ritmo]
 SR --> R[Predições e incertezas]
 AFIB --> R
 BRAD --> R
 TACH --> R
 O --> R
```

**O que falta para uso clínico:** avaliação confiável de ondas P, delineação QRS e PR, verificação manual de arritmias, avaliação de erros do detector, casos com marcapasso, comparação entre derivações e validação externa. Os parâmetros de variabilidade em 10 segundos são limitados; não equivalem a medidas clássicas de HRV de longa duração. Na análise estratificada, investigar idade, sexo e qualidade de derivação, mas evitar impor limites sem diretriz apropriada.

**Conclusão permitida:** cada SVM estima associação estatística com uma anotação de ritmo PTB-XL, não uma probabilidade clínica validada de arritmia.

## Conferência de exportação após o treinamento

A célula abaixo não treina novamente. Ela lista os arquivos produzidos e verifica se a contagem de `.joblib` corresponde ao inventário. Os modelos são gerados **somente após executar a célula de treinamento**.

In [10]:
import json
pasta_modelos = MODELOS_DIR
inventario_path = pasta_modelos / 'inventario_modelos.json'
if not inventario_path.exists():
    print('Ainda não existem modelos exportados. Execute a célula de treinamento acima.')
else:
    inventario = json.loads(inventario_path.read_text(encoding='utf-8'))
    esperados = inventario['arquivos_modelo']
    existentes = sorted(p.name for p in pasta_modelos.glob('*.joblib'))
    ausentes = sorted(set(esperados) - set(existentes))
    pacote = pasta_modelos / 'modelos_svm_modelos_02_completos.zip'
    print('Treinados:', inventario['quantidade_treinados'])
    print('Não treinados:', inventario['quantidade_nao_treinados'])
    print('Arquivos:', *esperados, sep='\n - ')
    print('Ausentes:', ausentes)
    print('ZIP existe:', pacote.exists())
    assert not ausentes, 'A exportação está incompleta.'
    assert pacote.exists(), 'O pacote ZIP não foi gerado.'

Treinados: 9
Não treinados: 3
Arquivos:
 - AFIB.joblib
 - AFLT.joblib
 - BIGU.joblib
 - PACE.joblib
 - SARRH.joblib
 - SBRAD.joblib
 - SR.joblib
 - STACH.joblib
 - SVARR.joblib
Ausentes: []
ZIP existe: True


## Avaliação de idade/sexo e responsabilidade dos filtros no visualizador

**Auditoria de sensibilidade por grupos:** a próxima célula avalia, nos mesmos registros do fold 10, a sensibilidade dos classificadores treinados para cada faixa etária e sexo informado. Os valores só são produzidos após o treinamento real.

**Integração com a interface:** ao modificar os campos existentes de idade/sexo, o visualizador deve **reconstruir o mesmo vetor com `contextualize_features`**, preservar as características do sinal original, usar os nomes/ordem de `features` salvos em cada `.joblib` e aplicar `decision_function` com o `threshold` armazenado. A saída de `decision_function` não é porcentagem nem probabilidade calibrada.

```mermaid
flowchart TD
    A[ECG registrado: sinal fixo] --> B[Extrair características das 12 derivações]
    C[Idade e sexo selecionados na interface] --> D[Aplicabilidade contextual das regras]
    B --> E[Vetor de características na ordem do treinamento]
    D --> E
    E --> F[Carregar cada SVM joblib]
    F --> G[decision_function + limiar salvo]
    G --> H[Atualizar predições sem modificar o traçado]
    D --> I[Reavaliar regras clínicas independentemente]
    I --> H
```

**Não misturar:** grupo de referência meramente visual, idade/sexo efetivamente escolhidos para análise e dados originais devem ser tratados como estados distintos. Se idade ou sexo forem omitidos, usar os metadados originais. Nenhuma alteração de filtro deve alterar os dados WFDB ou os rótulos de treinamento.

In [11]:

# Avaliação estratificada real dos modelos exportados no fold 10.
# Não altera modelos, thresholds nem os dados de teste.
test=merged.loc[merged.strat_fold==10].copy()
rows_group=[]
for label in labels:
    path=MODELOS_DIR/f"{label}.joblib"
    if not path.exists(): continue
    obj=joblib.load(path)
    model=obj["model"]
    cols=obj["features"]
    scores=model.decision_function(test[cols].to_numpy(dtype=float))
    pred=(scores>=obj["threshold"]).astype(int)
    truth=test["categories"].map(lambda s:int(label in s)).to_numpy() if "GROUPS" in globals() else test["code_set"].map(lambda s:int(label in s)).to_numpy()
    ages=pd.to_numeric(test["age"],errors="coerce")
    sexs=pd.to_numeric(test["sex_ptbxl"],errors="coerce")
    groups=ages.map(age_group_label)
    for axis,series in [("faixa",groups),("sexo",sexs.map({0:"masculino",1:"feminino"}).fillna("desconhecido"))]:
        for group in sorted(series.dropna().unique()):
            mask=(series==group).to_numpy()
            npos=int(truth[mask].sum())
            rows_group.append({"rotulo":label,"eixo":axis,"grupo":str(group),"n":int(mask.sum()),
                               "positivos":npos,
                               "sensibilidade":float(recall_score(truth[mask],pred[mask],zero_division=0)) if npos else np.nan,
                               "f1_positivo":float(f1_score(truth[mask],pred[mask],zero_division=0)) if npos else np.nan})
audit=pd.DataFrame(rows_group)
audit.to_csv(MODELOS_DIR/"avaliacao_por_idade_sexo.csv",index=False)
print("Auditoria estratificada exportada:",MODELOS_DIR/"avaliacao_por_idade_sexo.csv")
display(audit.head(30))


Auditoria estratificada exportada: C:\Users\Matheus\Documents\Modelos\modelos_02\avaliacao_por_idade_sexo.csv


,rotulo,eixo,grupo,n,positivos,sensibilidade,f1_positivo
0,AFIB,faixa,adulto_40_59,627,14,0.714286,0.689655
1,AFIB,faixa,adulto_jovem_17_39,262,1,0.000000,0.000000
2,AFIB,faixa,idoso_60_mais,1220,129,0.759690,0.671233
3,AFIB,faixa,nao_informado,34,8,0.750000,0.666667
4,AFIB,faixa,pediatrico_0_16,4,0,NaN,NaN
5,AFIB,sexo,feminino,1046,71,0.704225,0.680272
6,AFIB,sexo,masculino,1101,81,0.790123,0.663212
7,AFLT,faixa,adulto_40_59,627,0,NaN,NaN
8,AFLT,faixa,adulto_jovem_17_39,262,0,NaN,NaN
9,AFLT,faixa,idoso_60_mais,1220,7,0.285714,0.333333
